# roketsan undersample test: drop car-heavy imgs, no extra aug, no patches
yolo26m 1280, test-like val (seed 42). VARIANT='carunder': imgs w/ car share >= CU_SH and no bus/truck dropped (keep CU_KEEP)
default CU_SH 0.8 -> drops 1537 of 6471 imgs (24%): -27k cars, -2.2k vans, 0 truck/bus. car:van 5.5 -> 4.7, car:truck 10.3 -> 8.0, car:bus 22.3 -> 17.4
image-level ceiling: dropping every img w/o bus/truck only reaches car:bus 16.7. epochs scaled up so iterations match 20 ep full data. outputs -> roketsan_runs/p3_<variant>_e<EP>/

In [ ]:
# setup, rerun after runtime restart
!pip -q install -U ultralytics albumentations
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/data/train/annotations.csv'):
    !cp "/content/drive/MyDrive/data.zip" /content/data.zip && unzip -q /content/data.zip -d /content/data && rm /content/data.zip
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics;print(ultralytics.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 10.7 MB/s eta 0:00:00
Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
8.4.163


In [ ]:
import os,random,shutil,glob,json,time
import numpy as np,pandas as pd
from PIL import Image
DATA='/content/data'
WORK='/content/yolo'
RUNS='/content/drive/MyDrive/roketsan_runs'   # on drive, survives disconnect
CLASSES=['car','van','truck','bus']
C2I={c:i for i,c in enumerate(CLASSES)}
SEED=42
os.makedirs(RUNS,exist_ok=True)

## convert csv -> yolo labels, test-like val (same res mix as test, no 2000x1500), same for all teammates

In [ ]:
ann=pd.read_csv(f'{DATA}/train/annotations.csv')
imgs=sorted(f[:-4] for f in os.listdir(f'{DATA}/train/images'))
os.makedirs(f'{WORK}/images',exist_ok=True);os.makedirs(f'{WORK}/labels',exist_ok=True)
sizes={}
for i in imgs:
    sizes[i]=Image.open(f'{DATA}/train/images/{i}.jpg').size
g=ann.groupby('image_id')
for i in imgs:
    dst=f'{WORK}/images/{i}.jpg'
    if not os.path.exists(dst):
        os.symlink(f'{DATA}/train/images/{i}.jpg',dst)
    W,H=sizes[i]
    lines=[]
    if i in g.groups:
        for r in g.get_group(i).itertuples():
            cx=(r.x+r.w/2)/W;cy=(r.y+r.h/2)/H
            lines.append(f'{C2I[r.label]} {cx:.6f} {cy:.6f} {r.w/W:.6f} {r.h/H:.6f}')
    # empty file for bg imgs -> negatives, full of pedestrians
    open(f'{WORK}/labels/{i}.txt','w').write('\n'.join(lines))

# test-like val: split isnt random (0 test imgs at 2000x1500), so val copies test res mix
tsz=pd.Series([Image.open(f'{DATA}/test/images/{f}').size for f in os.listdir(f'{DATA}/test/images')]).value_counts(normalize=True)
bysz={}
for i in imgs:
    bysz.setdefault(sizes[i],[]).append(i)
random.seed(SEED);val_ids=[]
nval=int(len(imgs)*0.1)
for sz,share in tsz.items():
    pool=bysz.get(sz,[])
    k=min(len(pool)//3,round(nval*share))
    val_ids+=random.sample(pool,k)
    print(sz,'test share',round(share,3),'val',k,'of',len(pool))
val_ids=sorted(val_ids);vs=set(val_ids);tr_ids=[i for i in imgs if i not in vs]
open(f'{WORK}/train.txt','w').write('\n'.join(f'{WORK}/images/{i}.jpg' for i in tr_ids))
open(f'{WORK}/val.txt','w').write('\n'.join(f'{WORK}/images/{i}.jpg' for i in val_ids))
open(f'{WORK}/data.yaml','w').write(f"path: {WORK}\ntrain: train.txt\nval: val.txt\nnames: {CLASSES}\n")
json.dump(val_ids,open(f'{RUNS}/val_ids.json','w'))
print(len(tr_ids),len(val_ids),ann[ann.image_id.isin(val_ids)].label.value_counts().to_dict())

(1400, 788) test share 0.441 val 285 of 1299
(1360, 765) test share 0.263 val 170 of 743
(1400, 1050) test share 0.129 val 84 of 2498
(960, 540) test share 0.073 val 47 of 250
(1916, 1078) test share 0.071 val 46 of 537
(1920, 1080) test share 0.024 val 15 of 339
5824 647 {'car': 10811, 'van': 2423, 'truck': 1212, 'bus': 926}


## extra pixel aug: motion blur + night darkening, toggle with AUG_EXTRA

In [ ]:
import albumentations as A
from ultralytics.data import augment
AUG_EXTRA=False   # set per run in full-run cell
_extra=A.Compose([
    A.MotionBlur(blur_limit=(3,11),p=0.2),
    A.OneOf([A.RandomBrightnessContrast(brightness_limit=(-0.4,0.1),contrast_limit=0.2),
             A.RandomGamma(gamma_limit=(90,180))],p=0.2),
    A.GaussNoise(std_range=(0.01,0.04),p=0.1),
    A.ImageCompression(quality_range=(50,95),p=0.1),
])
if not hasattr(augment.Albumentations,'_orig_call'):
    augment.Albumentations._orig_call=augment.Albumentations.__call__
def _call(self,labels):
    labels=self._orig_call(labels)
    # pixel-only, boxes untouched, only in train pipeline
    if AUG_EXTRA:
        labels['img']=_extra(image=labels['img'])['image']
    return labels
augment.Albumentations.__call__=_call

## data variants: train list builder. ops joined by '+', e.g. 'res+bus2'

In [ ]:
# ops:
#  bus2 / bus3   imgs w/ bus repeated x2 / x3                  (oversample)
#  rare2         imgs w/ bus or truck x2                       (oversample)
#  under         imgs w/o bus/truck kept at 50% (bg imgs kept)  (undersample)
#  res           test-like resolutions repeated (ratio 1..3)   (resolution oversample)
#  resunder      over-represented res (1400x1050, 2000x1500) kept at 50% (resolution undersample)
#  crops         +1 img per rare img: 800px window around a bus/truck, rest gray, same scale (object-centric)
#  night         motion blur + darkening aug (AUG_EXTRA)
#  clsw          rare-class pos_weight in cls loss (not data, control)
#  minaug        + modified copies of bus (x2) / truck (x1) imgs, see minaug cell
#  patch         + van/truck/bus patches (margin, cars grayed out -> car count unchanged) x N_COPIES, gray canvases, see patch cell
#  carunder      imgs w/ car share >= CU_SH and no bus/truck kept at CU_KEEP (0 = dropped), empty bg imgs kept
#  clahe         dark imgs (mean gray <DARK_T) get clahe on L channel, train + val + test, bright imgs untouched
import torch,torch.nn as nn,cv2
from ultralytics.utils import loss as L
CLS_POS_W=None
if not hasattr(L.v8DetectionLoss,'_orig_init'):
    L.v8DetectionLoss._orig_init=L.v8DetectionLoss.__init__
def _init(self,model,*a,**k):
    self._orig_init(model,*a,**k)
    # yolo26 builds 2 of these (o2m + o2o head), both get weights
    if CLS_POS_W is not None:
        dev=next(model.parameters()).device
        self.bce=nn.BCEWithLogitsLoss(reduction='none',pos_weight=torch.tensor(CLS_POS_W,device=dev,dtype=torch.float32))
L.v8DetectionLoss.__init__=_init

CU_SH=0.8     # car share threshold. .8 -> 1537 imgs, -27k cars, -2.2k vans, car:van 4.7 car:bus 17.4. 1.0 = car-only (593, 0 vans lost, car:bus 21.3)
CU_KEEP=0.0   # fraction of those imgs kept
ENH=False   # set by variant 'clahe', predict() uses it too
DARK_T=50   # val: <50 gray = 14% of imgs, mAP .79 vs .83 bright
def enh(im):
    if not ENH or cv2.cvtColor(im,cv2.COLOR_BGR2GRAY).mean()>=DARK_T:
        return im
    l,a,b=cv2.split(cv2.cvtColor(im,cv2.COLOR_BGR2LAB))
    l=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge([l,a,b]),cv2.COLOR_LAB2BGR)

def make_clahe_dir():
    # dark imgs rewritten enhanced, rest symlinked, labels symlinked
    d=f'{WORK}/clahe';os.makedirs(f'{d}/images',exist_ok=True);os.makedirs(f'{d}/labels',exist_ok=True);n=0
    for i in imgs:
        p=f'{d}/images/{i}.jpg'
        if not os.path.exists(p):
            im=cv2.imread(f'{DATA}/train/images/{i}.jpg');e=enh(im)
            if e is im:
                os.symlink(f'{DATA}/train/images/{i}.jpg',p)
            else:
                cv2.imwrite(p,e,[cv2.IMWRITE_JPEG_QUALITY,95]);n+=1
        if not os.path.exists(f'{d}/labels/{i}.txt'):
            os.symlink(f'{WORK}/labels/{i}.txt',f'{d}/labels/{i}.txt')
    print('clahe dir, enhanced now',n)
    return d

def make_crops(size=800):
    # window around a bus/truck pasted back at its spot on a gray full-size canvas
    # -> same pixel scale as the real image (no upscaling by letterbox), cars outside window gone
    os.makedirs(f'{WORK}/crops',exist_ok=True)
    rng=random.Random(SEED);out=[];A=ann[ann.image_id.isin(tr_ids)];g=A.groupby('image_id')
    for i,R in A[A.label.isin(['bus','truck'])].groupby('image_id'):
        W,H=sizes[i];cw,ch=min(size,W),min(size,H)
        r=R.iloc[rng.randrange(len(R))]
        cx=r.x+r.w/2+rng.uniform(-.3,.3)*cw;cy=r.y+r.h/2+rng.uniform(-.3,.3)*ch
        x0=int(np.clip(cx-cw/2,0,W-cw));y0=int(np.clip(cy-ch/2,0,H-ch))
        lines=[]
        for b in g.get_group(i).itertuples():
            x1,y1=max(b.x,x0),max(b.y,y0);x2,y2=min(b.x+b.w,x0+cw),min(b.y+b.h,y0+ch)
            if x2<=x1 or y2<=y1:
                continue
            # keep if mostly visible, cut-off objects stay unlabeled like the source data
            if (x2-x1)*(y2-y1)<0.6*b.w*b.h or (x2-x1)*(y2-y1)<200:
                continue
            lines.append(f'{C2I[b.label]} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}')
        p=f'{WORK}/crops/{i}_c.jpg'
        if not os.path.exists(p):
            im=cv2.imread(f'{DATA}/train/images/{i}.jpg');can=np.full_like(im,114)
            can[y0:y0+ch,x0:x0+cw]=im[y0:y0+ch,x0:x0+cw]
            cv2.imwrite(p,can,[cv2.IMWRITE_JPEG_QUALITY,95])
        open(f'{WORK}/crops/{i}_c.txt','w').write('\n'.join(lines))
        out.append(p)
    return out

def build_variant(run,variant):
    global AUG_EXTRA,CLS_POS_W,ENH
    ops=set(variant.split('+'))-{'base'}
    AUG_EXTRA='night' in ops
    ENH='clahe' in ops
    IMD=make_clahe_dir()+'/images' if ENH else f'{WORK}/images'
    CLS_POS_W=None
    if 'clsw' in ops:
        n=ann[ann.image_id.isin(tr_ids)].label.value_counts()
        CLS_POS_W=np.clip((n['car']/n[CLASSES])**0.5,1,3).tolist()
    rng=random.Random(SEED)
    bus=set(ann[ann.label=='bus'].image_id);rare=set(ann[ann.label.isin(['bus','truck'])].image_id)
    lab=set(ann.image_id)
    cnt=ann.pivot_table(index='image_id',columns='label',values='x',aggfunc='count',fill_value=0).reindex(columns=CLASSES,fill_value=0)
    carheavy=set(cnt[(cnt.car/cnt.sum(axis=1)>=CU_SH)&(cnt.bus==0)&(cnt.truck==0)].index)
    trs=pd.Series([sizes[i] for i in tr_ids]).value_counts(normalize=True)
    ratio={s:tsz.get(s,0)/trs[s] for s in trs.index}
    rep={}
    for i in tr_ids:
        r=1.0
        if 'bus2' in ops and i in bus:
            r*=2
        if 'bus3' in ops and i in bus:
            r*=3
        if 'rare2' in ops and i in rare:
            r*=2
        if 'res' in ops:
            r*=np.clip(round(ratio[sizes[i]]),1,3)
        if 'under' in ops and i in lab and i not in rare:
            r*=0.5
        if 'resunder' in ops and ratio[sizes[i]]<0.5:
            r*=0.5
        if 'carunder' in ops and i in carheavy:
            r*=CU_KEEP
        # fractional -> stochastic rounding, seeded
        k=int(r)+(rng.random()<r-int(r))
        rep[i]=k
    lines=[f'{IMD}/{i}.jpg' for i in tr_ids for _ in range(rep[i])]
    open(f'{WORK}/val_{run}.txt','w').write('\n'.join(f'{IMD}/{i}.jpg' for i in val_ids))
    if 'crops' in ops:
        lines+=make_crops()
    if 'minaug' in ops:
        lines+=build_minaug()
    if 'patch' in ops:
        lines+=build_patches()
    open(f'{WORK}/train_{run}.txt','w').write('\n'.join(lines))
    open(f'{WORK}/data_{run}.yaml','w').write(f"path: {WORK}\ntrain: train_{run}.txt\nval: val_{run}.txt\nnames: {CLASSES}\n")
    sub=ann[ann.image_id.isin(tr_ids)].assign(r=lambda d:d.image_id.map(rep))
    bx=sub.groupby('label').r.sum().reindex(CLASSES)
    print(variant,'| imgs',len(tr_ids),'->',len(lines),'| boxes',bx.to_dict(),'| car:bus %.1f'%(bx['car']/bx['bus']),
          '| night',AUG_EXTRA,'| clahe',ENH,'| clsw',None if CLS_POS_W is None else np.round(CLS_POS_W,2).tolist())
    return f'{WORK}/data_{run}.yaml',len(lines)/len(tr_ids)

def train_run(run,variant,model='yolo26m.pt',imgsz=1280,epochs=40,batch=16,close=10):
    data_yaml,grow=build_variant(run,variant)
    ep=max(5,round(epochs/grow))   # same iterations for every variant, fair compare
    import psutil
    need=len(tr_ids)*grow*imgsz*imgsz*0.75*3/1e9
    cache='ram' if need<0.7*psutil.virtual_memory().total/1e9 else False
    print(run,'epochs',ep,'ram need ~%.0fGB'%need,'cache',cache)
    YOLO(model).train(data=data_yaml,epochs=ep,imgsz=imgsz,batch=batch,
        cos_lr=True,close_mosaic=min(close,ep//3),mosaic=1.0,mixup=0.0,copy_paste=0.0,
        scale=0.5,fliplr=0.5,flipud=0.0,hsv_v=0.5,degrees=0.0,
        max_det=500,patience=100,workers=8,cache=cache,
        project=RUNS,name=run,exist_ok=True,save_period=5,seed=SEED,deterministic=True)
    return f'{RUNS}/{run}/weights/best.pt'

## minority patch factory: crop each van/truck/bus w/ margin, N modified copies, packed on gray test-size canvases
cars + half-visible vehicles in the margin are painted gray -> no extra car boxes, no unlabeled vehicle pixels
N_COPIES (per class) / MARGIN / AUG_TYPES are the knobs. saved to drive zip per setting, teammates reuse it

In [ ]:
import albumentations as A,cv2,zipfile,shutil
N_COPIES={'van':1,'truck':2,'bus':3}   # modified copies per object, class left out -> not patched
PATCH_CLASSES=list(N_COPIES)
MARGIN=0.3                   # context around box, fraction of box size (min 16px)
AUG_TYPES=['flip','scale','bright','night','hue','blur','noise','jpeg']   # drop any you dont want
CANVAS=(1400,788)            # most common test res -> letterbox scales patches like test imgs

def _patch_aug(types):
    T=[]
    if 'flip' in types:
        T.append(A.HorizontalFlip(p=0.5))
    if 'scale' in types:
        T.append(A.RandomScale(scale_limit=0.2,p=0.7))
    photo=[]
    if 'bright' in types:
        photo.append(A.RandomBrightnessContrast(brightness_limit=0.25,contrast_limit=0.25))
    if 'night' in types:
        photo.append(A.RandomGamma(gamma_limit=(130,220)))   # gamma>1 darkens
    if 'hue' in types:
        photo.append(A.HueSaturationValue(hue_shift_limit=8,sat_shift_limit=25,val_shift_limit=20))
    if photo:
        T.append(A.OneOf(photo,p=0.8))
    if 'blur' in types:
        T.append(A.MotionBlur(blur_limit=(3,7),p=0.25))
    if 'noise' in types:
        T.append(A.GaussNoise(std_range=(0.01,0.04),p=0.15))
    if 'jpeg' in types:
        T.append(A.ImageCompression(quality_range=(60,95),p=0.3))
    return A.Compose(T,bbox_params=A.BboxParams(format='pascal_voc',label_fields=['cls'],min_visibility=0.6))

def build_patches():
    key=f"{'-'.join(f'{c}{k}' for c,k in N_COPIES.items())}_nocar_m{MARGIN}_{'-'.join(sorted(AUG_TYPES))}{'_clahe' if ENH else ''}"
    out=f'{WORK}/patch_{key}';zp=f'{RUNS}/patch_{key}.zip'
    if os.path.exists(zp):
        print('found',zp,'-> unzip');shutil.rmtree(out,ignore_errors=True);zipfile.ZipFile(zp).extractall(WORK)
    else:
        os.makedirs(f'{out}/images',exist_ok=True);os.makedirs(f'{out}/labels',exist_ok=True)
        aug=_patch_aug(AUG_TYPES);random.seed(SEED);np.random.seed(SEED)
        A_tr=ann[ann.image_id.isin(tr_ids)];g=A_tr.groupby('image_id');P=[]
        for i,T in A_tr[A_tr.label.isin(PATCH_CLASSES)].groupby('image_id'):
            # dark src img -> clahe first when ENH, so patches match the enhanced train imgs
            im=enh(cv2.imread(f'{DATA}/train/images/{i}.jpg'));H,W=im.shape[:2];G=g.get_group(i)
            for t in T.itertuples():
                mx=max(16,MARGIN*t.w);my=max(16,MARGIN*t.h)
                x0,y0=int(max(0,t.x-mx)),int(max(0,t.y-my));x1,y1=int(min(W,t.x+t.w+mx)),int(min(H,t.y+t.h+my))
                crop=im[y0:y1,x0:x1].copy();bb,cl,drop=[],[],[]
                for b in G.itertuples():
                    ax1,ay1=max(b.x,x0),max(b.y,y0);ax2,ay2=min(b.x+b.w,x1),min(b.y+b.h,y1)
                    if ax2<=ax1 or ay2<=ay1:
                        continue
                    r=[int(ax1-x0),int(ay1-y0),int(np.ceil(ax2-x0)),int(np.ceil(ay2-y0))]
                    # keep non-car vehicles mostly inside, gray out cars + cut-off ones
                    if b.label!='car' and (ax2-ax1)*(ay2-ay1)>=0.6*b.w*b.h:
                        bb.append([ax1-x0,ay1-y0,ax2-x0,ay2-y0]);cl.append(b.label)
                    else:
                        drop.append(r)
                keep=crop.copy()
                for a,b_,c,d in drop:
                    crop[b_:d,a:c]=114
                # restore kept boxes, occluding car corner stays visible
                for a,b_,c,d in bb:
                    a,b_,c,d=int(a),int(b_),int(np.ceil(c)),int(np.ceil(d));crop[b_:d,a:c]=keep[b_:d,a:c]
                for _ in range(N_COPIES[t.label]):
                    o=aug(image=crop,bboxes=np.array(bb,float),cls=cl)
                    ob=[(b,c) for b,c in zip(o['bboxes'],o['cls']) if (b[2]-b[0])*(b[3]-b[1])>=200]
                    if any(c in PATCH_CLASSES for _,c in ob):
                        P.append((o['image'],ob))
        random.shuffle(P)
        # shelf packing, native scale, 8px gray gap
        CW,CH=CANVAS;gap=8;n=0;can=None
        def flush(can,labs,n):
            cv2.imwrite(f'{out}/images/p{n:05d}.jpg',can,[cv2.IMWRITE_JPEG_QUALITY,92])
            open(f'{out}/labels/p{n:05d}.txt','w').write('\n'.join(labs))
        for pim,ob in P:
            ph,pw=pim.shape[:2]
            if pw>CW or ph>CH:
                continue
            if can is None:
                can=np.full((CH,CW,3),114,np.uint8);x=y=rh=0;labs=[]
            if x+pw>CW:
                x=0;y+=rh+gap;rh=0
            if y+ph>CH:
                flush(can,labs,n);n+=1;can=np.full((CH,CW,3),114,np.uint8);x=y=rh=0;labs=[]
            can[y:y+ph,x:x+pw]=pim
            for (bx1,by1,bx2,by2),c in ob:
                labs.append(f'{C2I[c]} {(x+(bx1+bx2)/2)/CW:.6f} {(y+(by1+by2)/2)/CH:.6f} {(bx2-bx1)/CW:.6f} {(by2-by1)/CH:.6f}')
            x+=pw+gap;rh=max(rh,ph)
        if can is not None:
            flush(can,labs,n)
        with zipfile.ZipFile('/content/patch.zip','w',zipfile.ZIP_STORED) as z:
            for f in glob.glob(f'{out}/*/*'):
                z.write(f,os.path.relpath(f,WORK))
        shutil.copy('/content/patch.zip',zp)
    files=sorted(glob.glob(f'{out}/images/*.jpg'))
    cnt=pd.Series([int(l.split()[0]) for f in glob.glob(f'{out}/labels/*.txt') for l in open(f).read().split('\n') if l]).value_counts()
    print('patch canvases',len(files),'| boxes',{CLASSES[k]:int(v) for k,v in cnt.sort_index().items()})
    return files

## local metric: per-class AP@0.5 (all-point interp), class mean. gt <200px² ignored, preds filtered at pred_min

In [ ]:
def iou_1n(b,B):
    # b: x,y,w,h ; B: n x 4
    x1=np.maximum(b[0],B[:,0]);y1=np.maximum(b[1],B[:,1])
    x2=np.minimum(b[0]+b[2],B[:,0]+B[:,2]);y2=np.minimum(b[1]+b[3],B[:,1]+B[:,3])
    inter=np.clip(x2-x1,0,None)*np.clip(y2-y1,0,None)
    return inter/(b[2]*b[3]+B[:,2]*B[:,3]-inter+1e-9)

def evaluate(pred,gt,pred_min=0,verbose=True):
    # pred/gt: df image_id,label,conf(pred only),x,y,w,h
    # pred_min=0 -> host scores all preds (strict), pred_min=200 -> host drops small preds before scoring
    gt=gt[gt.w*gt.h>=200];pred=pred[pred.w*pred.h>=pred_min]
    aps={}
    for c in CLASSES:
        G=gt[gt.label==c];P=pred[pred.label==c].sort_values('conf',ascending=False)
        gb={k:v[['x','y','w','h']].values.astype(float) for k,v in G.groupby('image_id')}
        used={k:np.zeros(len(v),bool) for k,v in gb.items()}
        tp=np.zeros(len(P))
        for j,r in enumerate(P.itertuples()):
            B=gb.get(r.image_id)
            if B is None:
                continue
            io=iou_1n(np.array([r.x,r.y,r.w,r.h],float),B)
            io[used[r.image_id]]=-1
            k=io.argmax()
            if io[k]>=0.5:
                tp[j]=1;used[r.image_id][k]=True
        if len(G)==0:
            aps[c]=np.nan;continue
        ctp=np.cumsum(tp);rec=ctp/len(G);prec=ctp/np.arange(1,len(P)+1)
        mrec=np.r_[0,rec,1];mpre=np.r_[0,prec,0]
        for i in range(len(mpre)-2,-1,-1):
            mpre[i]=max(mpre[i],mpre[i+1])
        idx=np.where(mrec[1:]!=mrec[:-1])[0]
        aps[c]=float(np.sum((mrec[idx+1]-mrec[idx])*mpre[idx+1]))
    m=np.nanmean(list(aps.values()))
    if verbose:
        print(f'mAP50 {m:.4f}',{k:round(v,4) for k,v in aps.items()})
    return m,aps

gt_val=ann[ann.image_id.isin(val_ids)].copy()

## predict + postprocess + submission writer

In [ ]:
from ultralytics import YOLO
import ultralytics.utils.nms as _NMS
if not hasattr(_NMS,'_orig_nms'):
    _NMS._orig_nms=_NMS.non_max_suppression
    # multi-label: one box can be car and van at once, +1.8 mAP on run01 val vs e2e+dup
    _NMS.non_max_suppression=lambda *a,**k:_NMS._orig_nms(*a,**{**k,'multi_label':True})

def predict(weights,ids,img_dir,imgsz=1280,tta=False,conf=0.001,iou=0.6,max_det=1000,batch=16):
    m=YOLO(weights);rows=[]
    for s in range(0,len(ids),batch):
        chunk=ids[s:s+batch]
        src=[enh(cv2.imread(f'{img_dir}/{i}.jpg')) for i in chunk] if ENH else [f'{img_dir}/{i}.jpg' for i in chunk]
        res=m.predict(src,imgsz=imgsz,conf=conf,iou=iou,
                      max_det=max_det,augment=tta,half=True,verbose=False,nms=True)   # o2m head + nms
        for i,r in zip(chunk,res):
            b=r.boxes
            if len(b)==0:
                continue
            xyxy=b.xyxy.cpu().numpy();cf=b.conf.cpu().numpy();cl=b.cls.cpu().numpy().astype(int)
            for (x1,y1,x2,y2),p,k in zip(xyxy,cf,cl):
                rows.append((i,CLASSES[k],float(p),x1,y1,x2-x1,y2-y1))
    return pd.DataFrame(rows,columns=['image_id','label','conf','x','y','w','h'])

SWAP={'car':'van','van':'car','truck':'bus','bus':'truck'}
def add_dups(p,alpha=0.0):   # superseded by multi-label nms, keep 0
    # confusable class copy at lower conf, keeps recall on car<->van, truck<->bus
    if alpha<=0:
        return p
    d=p.copy();d['label']=d.label.map(SWAP);d['conf']=d.conf*alpha
    return pd.concat([p,d],ignore_index=True)

MIN_PRED_AREA=100   # <100 can never hit a >=200 gt at iou .5, 100-200 still can
def write_sub(p,path,min_area=None):
    p=p[p.w*p.h>=(MIN_PRED_AREA if min_area is None else min_area)]
    sub=pd.read_csv(f'{DATA}/sample_submission.csv')
    s={k:' '.join(f'{r.label} {r.conf:.5f} {r.x:.1f} {r.y:.1f} {r.w:.1f} {r.h:.1f}' for r in v.itertuples())
       for k,v in p.groupby('image_id')}
    sub['PredictionString']=sub.image_id.map(s).fillna('none')
    sub.to_csv(path,index=False)
    print(path,len(sub),'none rows:',(sub.PredictionString=='none').sum())
    return sub

## train + val score. carunder: fewer imgs -> more epochs, same iterations as 20 ep full data

In [ ]:
VARIANT='carunder'      # 'carunder' | 'base' | combos e.g. 'carunder+patch'
EP=20                   # nominal, carunder keeps ~76% of imgs -> ~26 real ep, same iterations
IMGSZ=1280              # 1536 = teammate res (their LB 0.781), batch drops to 8, ~1.45x slower
RUN='p3_'+VARIANT.replace('+','_')+f'_sh{CU_SH}_e{EP}'+('' if IMGSZ==1280 else f'_{IMGSZ}')   # -> p3_carunder_sh0.8_e20(_1536)
W=train_run(RUN,VARIANT,model='yolo26m.pt',imgsz=IMGSZ,epochs=EP,batch=16 if IMGSZ<=1280 else 8,close=5)
pv=predict(W,val_ids,f'{DATA}/train/images',imgsz=IMGSZ)
pv.to_parquet(f'{RUNS}/{RUN}/val_pred.parquet')
m,aps=evaluate(pv,gt_val)
json.dump(dict(variant=VARIANT+('' if IMGSZ==1280 else f'_{IMGSZ}'),map50=m,**aps),open(f'{RUNS}/{RUN}/val_metrics.json','w'))

carunder | imgs 5824 -> 4424 | boxes {'car': 88942, 'van': 18387, 'truck': 10897, 'bus': 4684} | car:bus 19.0 | night False | clahe False | clsw None
p3_carunder_sh0.8_e20 epochs 26 ram need ~16GB cache ram
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo/data_p3_carunder_sh0.8_e20.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=26, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, imgsz=1280, io

resume after disconnect: rerun cells above except train, then

In [ ]:
# YOLO(f'{RUNS}/{RUN}/weights/last.pt').train(resume=True)

## compare all p2_/p3_ runs (same val split)

In [ ]:
rows=[json.load(open(f)) for f in glob.glob(f'{RUNS}/p[23]_*/val_metrics.json')]
T=pd.DataFrame(rows).drop_duplicates('variant',keep='last').set_index('variant')[['map50']+CLASSES]
if 'base' in T.index:
    T=pd.concat([T,(T-T.loc['base']).add_prefix('Δ')],axis=1)
T.round(4)

,map50,car,van,truck,bus
variant,,,,,
patch,0.8432,0.9186,0.7369,0.8177,0.8997
patch+clahe,0.8432,0.9200,0.7368,0.8168,0.8994
carunder,0.8446,0.9173,0.7408,0.8208,0.8996


## test preds + submission (multi-label nms, min area 100)

In [ ]:
test_ids=pd.read_csv(f'{DATA}/sample_submission.csv').image_id.tolist()
pt=predict(W,test_ids,f'{DATA}/test/images',imgsz=IMGSZ)
pt.to_parquet(f'{RUNS}/{RUN}/test_pred_raw.parquet')
write_sub(pt,f'{RUNS}/{RUN}/submission.csv')

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

,image_id,PredictionString
0,img_000001,car 0.92285 1158.1 604.6 157.2 129.6 van 0.902...
1,img_000002,truck 0.88965 398.4 0.3 217.3 133.3 car 0.8740...
2,img_000003,car 0.90820 27.0 281.6 59.2 67.8 van 0.90820 9...
3,img_000004,truck 0.89111 0.0 410.9 189.0 74.8 car 0.88623...
4,img_000008,bus 0.93115 342.3 651.3 148.2 94.1 bus 0.91602...
...,...,...
2113,img_008621,car 0.92432 912.7 673.6 121.1 71.2 car 0.92139...
2114,img_008622,car 0.86865 597.7 357.7 31.7 48.1 car 0.86426 ...
2115,img_008623,car 0.88574 482.4 371.9 56.3 74.4 car 0.88574 ...
2116,img_008626,car 0.90479 313.6 974.5 111.5 53.9 car 0.89648...


## TTA like teammate (their final: views base/flip/up, wbf iou .65). works on any finished run, no training
to use on an old run: run setup cells (up to "predict + postprocess") then jump here, skip train cell

In [13]:
!pip -q install ensemble-boxes
from ensemble_boxes import weighted_boxes_fusion
UP=1.25   # 'up' view = imgsz*UP, teammate's exact value unknown
def predict_tta(weights,ids,img_dir,imgsz=1280,views=('base','flip','up'),wbf_iou=0.65,batch=8):
    m=YOLO(weights);rows=[]
    for s in range(0,len(ids),batch):
        chunk=ids[s:s+batch]
        ims=[enh(cv2.imread(f'{img_dir}/{i}.jpg')) for i in chunk]
        outs=[]
        for v in views:
            src=[np.ascontiguousarray(im[:,::-1]) for im in ims] if v=='flip' else ims
            sz=int(round(imgsz*UP/32)*32) if v=='up' else imgsz
            outs.append(m.predict(src,imgsz=sz,conf=0.001,iou=0.6,max_det=1000,half=True,verbose=False,nms=True))
        for k,i in enumerate(chunk):
            H,W=ims[k].shape[:2];B,S,L=[],[],[]
            for v,res in zip(views,outs):
                b=res[k].boxes;xy=b.xyxy.cpu().numpy().copy()
                if v=='flip':
                    xy[:,[0,2]]=W-xy[:,[2,0]]
                B.append((xy/np.array([W,H,W,H])).clip(0,1));S.append(b.conf.cpu().numpy());L.append(b.cls.cpu().numpy())
            if not any(len(x) for x in S):
                continue
            bb,ss,ll=weighted_boxes_fusion(B,S,L,iou_thr=wbf_iou,skip_box_thr=0.001,conf_type='avg')
            for (x1,y1,x2,y2),p,cl in zip(bb,ss,ll):
                rows.append((i,CLASSES[int(cl)],float(p),x1*W,y1*H,(x2-x1)*W,(y2-y1)*H))
        if s%400==0:
            print(s,'/',len(ids))
    return pd.DataFrame(rows,columns=['image_id','label','conf','x','y','w','h'])

val first: tta must beat the run's val_metrics.json, else skip test tta

In [14]:
TTA_RUN='p3_carunder_sh0.8_e20'   # any finished run folder in RUNS
TSZ=1280                          # imgsz that run was trained at
TW=f'{RUNS}/{TTA_RUN}/weights/best.pt'
pvt=predict_tta(TW,val_ids,f'{DATA}/train/images',imgsz=TSZ)
pvt.to_parquet(f'{RUNS}/{TTA_RUN}/val_pred_tta.parquet')
print('no tta',json.load(open(f'{RUNS}/{TTA_RUN}/val_metrics.json'))['map50'])
m,aps=evaluate(pvt,gt_val)
json.dump(dict(variant=TTA_RUN+'_tta',map50=m,**aps),open(f'{RUNS}/{TTA_RUN}/val_metrics_tta.json','w'))

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
0 / 647
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.86062           1     0.96813           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.82375           1     0.88375           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [       0.82           1      0.8825           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.82875           0     0.88375           0].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [       0.83           0        0.97           0].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.78945           0     0.85469           0].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.82625           1     0.88125           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.85625           1     0.96875           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


...

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.14375           1     0.21602           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [   0.038789           1     0.13219           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.86187           1     0.96813           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [   0.032891           1     0.14602           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [   0.033555           1     0.14516           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [   0.036094           1     0.12859           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py

no tta 0.8446248422907441
mAP50 0.8598 {'car': 0.9266, 'van': 0.7613, 'truck': 0.8419, 'bus': 0.9094}


test tta + submission, only if val improved

In [15]:
test_ids=pd.read_csv(f'{DATA}/sample_submission.csv').image_id.tolist()
ptt=predict_tta(TW,test_ids,f'{DATA}/test/images',imgsz=TSZ)
ptt.to_parquet(f'{RUNS}/{TTA_RUN}/test_pred_tta.parquet')
write_sub(ptt,f'{RUNS}/{TTA_RUN}/submission_tta.csv')

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
0 / 2118
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.11563           1     0.21125           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.81812           1     0.88187           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.82875           0     0.88375           0].
  warnings.warn("Zero area box skipped: {}.".format(box_part))


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [   0.036152           1    0.083828           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.85688           1     0.96813           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [      0.775           1     0.81641           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.75742           1     0.79414           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


...

/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.85625           1      0.9725           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.14688           1      0.2125           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.70508           1     0.74883           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.71641           1     0.76172           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py:66: UserWarning: Zero area box skipped: [    0.69297           1     0.73516           1].
  warnings.warn("Zero area box skipped: {}.".format(box_part))
/usr/local/lib/python3.13/dist-packages/ensemble_boxes/ensemble_boxes_wbf.py

/content/drive/MyDrive/roketsan_runs/p3_carunder_sh0.8_e20/submission_tta.csv 2118 none rows: 1


,image_id,PredictionString
0,img_000001,car 0.91195 1158.0 604.8 158.3 129.5 van 0.902...
1,img_000002,truck 0.87858 397.8 1.2 217.5 134.1 truck 0.85...
2,img_000003,van 0.90885 935.2 517.5 102.8 57.2 car 0.90804...
3,img_000004,truck 0.89502 0.0 411.6 189.6 73.3 car 0.88623...
4,img_000008,bus 0.93197 341.8 651.8 148.3 94.6 bus 0.91846...
...,...,...
2113,img_008621,car 0.92204 912.9 673.3 121.0 71.6 car 0.91602...
2114,img_008622,car 0.87077 558.2 505.3 36.4 59.6 car 0.86540 ...
2115,img_008623,car 0.89144 1248.6 477.5 111.4 71.3 van 0.8769...
2116,img_008626,car 0.90495 313.9 974.8 112.0 53.0 car 0.90088...


In [16]:
from google.colab import files
files.download('/content/drive/MyDrive/roketsan_runs/p3_carunder_sh0.8_e20/submission_tta.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>